In [1]:
import numpy as np
import tensorflow as tf
import pickle

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [8]:
# Load IMDB RAW TEXT


num_words = 10000
max_len = 500

# Load raw dataset (not pre-encoded)
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=num_words)


d:\DL Project\venv\Lib\site-packages\numpy\lib\_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [9]:
word_index = imdb.get_word_index()

In [10]:
# Convert integer sequences back to words
reverse_word_index = {v: k for k, v in word_index.items()}

In [11]:
def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in encoded_review])

X_train_text = [decode_review(review) for review in X_train]
X_test_text = [decode_review(review) for review in X_test]


In [12]:
# Tokenizer 

tokenizer = Tokenizer(num_words=num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

In [13]:
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [14]:
# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [15]:
# Building the  Model 

model = Sequential([
    Embedding(num_words, 128, input_length=max_len),
    LSTM(128),
    Dense(1, activation="sigmoid")
])

d:\DL Project\venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [16]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [17]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [18]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 648s 1s/step - accuracy: 0.7779 - loss: 0.4586 - val_accuracy: 0.8602 - val_loss: 0.3558
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 795s 1s/step - accuracy: 0.8833 - loss: 0.2908 - val_accuracy: 0.8656 - val_loss: 0.3272
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 747s 1s/step - accuracy: 0.9294 - loss: 0.1905 - val_accuracy: 0.8448 - val_loss: 0.4194
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 535s 767ms/step - accuracy: 0.9406 - loss: 0.1561 - val_accuracy: 0.8536 - val_loss: 0.4066
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 343s 549ms/step - accuracy: 0.9574 - loss: 0.1180 - val_accuracy: 0.8622 - val_loss: 0.4702
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 311s 497ms/step - accuracy: 0.9697 - loss: 0.0858 - val_accuracy: 0.8630 - val_loss: 0.5253
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 328s 524ms/step - accuracy: 0.9811 - loss: 0.0578 - val_accuracy: 0.8434 - val_loss: 0.5809


In [19]:
model.save("sentiment_model.keras")

print("Training complete. Model and tokenizer saved.")

Training complete. Model and tokenizer saved.
